# Demo v0.2 — обратный расчёт ТЭП

Демонстрация трёх режимов работы модели:

| Режим | Функция | Когда использовать |
|---|---|---|
| **Обратный расчёт** | `solve_max_kit` | Найти максимально допустимый КИТ |
| **Проверочный расчёт** | `verify_kit` | Проверить заданный КИТ |
| **Сравнение сценариев** | `compare_scenarios` | Сравнить несколько вариантов рядом |

Все результаты несут **аудит-трейл**: значение, статус, источник норматива, формулу расчёта.  
Итог экспортируется в Excel (`output/tep_report.xlsx`) с цветовой маркировкой статусов.

In [1]:
import pathlib
import pandas as pd

from urban_model import solve_max_kit, verify_kit, compare_scenarios
from urban_model.models import Site, CalculationOptions, Scenario
from urban_model.normatives import load_normatives
from urban_model.export import to_xlsx, results_to_dataframe
from urban_model.modes.compare import run_scenarios

# Загрузка нормативной базы (Санкт-Петербург, наследует от russia.yaml)
norms = load_normatives('spb')
print(f"Профиль: {norms.profile}")

Профиль: spb


---
## 1. Обратный расчёт — найти максимальный КИТ

**Задача:** дан квартал площадью **3 га**, 12 этажей, с ППТ.  
Найти максимально допустимый КИТ, при котором баланс территории сходится.

In [2]:
site_3ga = Site(area_m2=30_000, name='Средний квартал 3 га')
opts_12fl = CalculationOptions(floors=12, planning_doc=True)

res_inv = solve_max_kit(site_3ga, opts_12fl, norms)
print(res_inv.summary())

Профиль: spb
КИТ:                     0.596 (норм. макс 2.5)
Площадь квартир:         13,403 м²
Население:               479 чел
Плотность:               159.6 чел/га [ok]
ДОО (мест):              требуется 29.200090680803573 → принято 30
СОШ (мест):              требуется 57.44280133928571 → принято 60
ЗНОП (м²/чел):           0 → итого 0 м²
Парковки требуется:      168 м/м
Открытые парковки:       21 м/м, 436 м²
Баланс:                  OK (+5 м²)
Ограничивающий фактор:   участки СОШ (13,800 м², 46.0% квартала)


### Аудит-трейл: источники и формулы каждого показателя

In [3]:
from urban_model.export.table import result_to_audit
import pandas as pd

df_audit = pd.DataFrame(result_to_audit(res_inv))
df_audit.style.set_properties(**{'text-align': 'left'}).set_table_styles(
    [{'selector': 'th', 'props': [('text-align', 'left')]}]
)

,поле,значение,ед.,статус,источник,формула
0,kit,0.595703,nan,ok,nan,вход в compute_tep_for_kit
1,kit_normative_max,2.500000,nan,ok,ПЗЗ СПб,"ПЗЗ СПб, ППТ=да"
2,gfa,17871.093750,m2,ok,nan,КИТ × S_квартала = 0.596 × 30000.0
3,apartments_area,13403.320312,m2,ok,принято условно,GFA × (1 − ВПП=0.0) × 0.75
4,population,478.690011,чел,ok,НГП СПб,S_квартир / 28
5,population_check_20,670.166016,чел,ok,СП 42.13330.2016,"S_квартир / 20 (формально, для проверки плотности)"
6,density_chel_per_ga,159.563337,чел/га,ok,СП 42.13330.2016,население / (S_квартала / 10000)
7,kindergarten_places_required,29.200091,мест,ok,НГП СПб,население × 61 / 1000
8,kindergarten_places_accepted,30.000000,мест,ok,nan,вверх кратно 5 → разбивка по объектам [160]
9,kindergarten_plot_area,6400.000000,m2,ok,nan,"Σ piecewise(plot_per_place, capacity) по объектам ДОО"


---
## 2. Проверочный расчёт — задать КИТ вручную

**Задача:** архитектор предлагает КИТ=1.6 на квартале 5 га, 15 этажей.  
Проверить, выполняются ли нормативы, и получить полный набор ТЭП.

In [4]:
site_5ga = Site(area_m2=50_000, name='Квартал 5 га')
opts_15fl = CalculationOptions(floors=15, planning_doc=True)

res_ver = verify_kit(1.6, site_5ga, opts_15fl, norms)
print(res_ver.summary())

# Статус по плотности
print(f"\nПлотность: {res_ver.density_chel_per_ga.value:.1f} чел/га "
      f"[{res_ver.density_chel_per_ga.status.value}] "
      f"(норматив ≤ {res_ver.density_chel_per_ga.normative})")

Профиль: spb
КИТ:                     1.600 (норм. макс 2.5)
Площадь квартир:         60,000 м²
Население:               2,143 чел
Плотность:               428.6 чел/га [error]
ДОО (мест):              требуется 130.7142857142857 → принято 135
СОШ (мест):              требуется 257.1428571428571 → принято 260
ЗНОП (м²/чел):           3 → итого 6,429 м²
Парковки требуется:      750 м/м
Открытые парковки:       94 м/м, 1,950 м²
Баланс:                  ДЕФИЦИТ (-18,612 м²)
  ⚠ Плотность 600 чел/га > норматива 450 (по 20 м²/чел)

Плотность: 428.6 чел/га [error] (норматив ≤ 450)


---
## 3. Сравнение сценариев

Сравниваем **4 варианта** одного квартала (10 га):  
- Обратный расчёт: с ППТ / без ППТ  
- Проверочный расчёт: КИТ=1.5 и КИТ=2.0

In [5]:
site_10ga = Site(area_m2=100_000, name='Крупный квартал 10 га')

scenarios = [
    Scenario(
        name='Макс. КИТ, с ППТ',
        site=site_10ga,
        options=CalculationOptions(floors=15, planning_doc=True),
    ),
    Scenario(
        name='Макс. КИТ, без ППТ',
        site=site_10ga,
        options=CalculationOptions(floors=15, planning_doc=False),
    ),
    Scenario(
        name='Проверка КИТ=1.5',
        site=site_10ga,
        options=CalculationOptions(floors=12),
        mode='verify',
        kit=1.5,
    ),
    Scenario(
        name='Проверка КИТ=2.0',
        site=site_10ga,
        options=CalculationOptions(floors=17),
        mode='verify',
        kit=2.0,
    ),
]

df_compare = compare_scenarios(scenarios, norms)
df_compare

сценарий,"Макс. КИТ, с ППТ","Макс. КИТ, без ППТ",Проверка КИТ=1.5,Проверка КИТ=2.0
показатель,,,,
КИТ,1.2,1.2,1.5,2.0
КИТ норм. макс.,2.5,1.4,2.5,2.5
"GFA, м²",119980.47,119941.41,150000.0,200000.0
"Площадь квартир, м²",89985.35,89956.05,112500.0,150000.0
"Население, чел",3213.76,3212.72,4017.86,5357.14
"Плотность, чел/га",321.38,321.27,401.79,535.71
Плотность статус,ok,ok,error,error
ДОО мест (требуется),196.04,195.98,245.09,326.79
ДОО мест (принято),200,200,250,330


---
## 4. Экспорт в Excel

Файл содержит два листа:
- **«Сравнение»** — сводная таблица КПЭ (строки — показатели, столбцы — сценарии)  
- **«Аудит»** — все поля всех сценариев с источниками и формулами

Ячейки со статусом **ok** окрашены зелёным, **warning** — жёлтым, **error** / **ДЕФИЦИТ** — красным.

In [6]:
# Запускаем все сценарии и получаем raw-результаты (нужны для to_xlsx)
pairs = run_scenarios(scenarios, norms)

# Создаём директорию и экспортируем
out_dir = pathlib.Path('../output')
out_dir.mkdir(exist_ok=True)

xlsx_path = to_xlsx(pairs, out_dir / 'tep_report.xlsx')
print(f'Файл записан: {xlsx_path.resolve()}')
print(f'Размер: {xlsx_path.stat().st_size:,} байт')

Файл записан: D:\Github\my_urban_model\output\tep_report.xlsx
Размер: 12,091 байт


---
## Итоги демо

| Что показано | Статус |
|---|---|
| Обратный расчёт (`solve_max_kit`) | ✅ |
| Проверочный расчёт (`verify_kit`) | ✅ |
| Сравнение сценариев (`compare_scenarios`) | ✅ |
| Аудит-трейл (source + formula на каждом поле) | ✅ |
| Экспорт в xlsx с цветовой маркировкой статусов | ✅ |

**Следующий шаг (v0.3):** расширение соцобъектов — поликлиники, ФОК, культура; парковочные сценарии (подземные + многоуровневые).